# HW08 Part B :: Anomaly Detection using Convolutional Autoencoder

COSC 6373 -- Adam Nelson-Archer, 2140122

## Prerequisites

- Install packages if needed:
  - `pip install -U tensorflow matplotlib numpy pandas scikit-learn kagglehub`
- Dataset reference (assignment):
  - https://www.kaggle.com/datasets/alxmamaev/flowers-recognition?select=flowers
- Reference method for reconstruction error + latent density scoring:
  - https://medium.com/@judewells/image-anomaly-detection-novelty-detection-using-convolutional-auto-encoders-in-keras-1c31321c10f2

In [ ]:
from pathlib import Path
import random

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.neighbors import KernelDensity

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

In [ ]:
# Download and locate the Flowers dataset from Kaggle.
download_root = Path(kagglehub.dataset_download("alxmamaev/flowers-recognition"))
cfg_data_root = download_root / "flowers"

cfg_img_size = (96, 96)
cfg_batch_size = 32
cfg_epochs = 15
cfg_learning_rate = 1e-3

# Part B requirement: train autoencoder on sunflower images.
cfg_normal_class = "sunflower"

if not cfg_data_root.exists():
    raise FileNotFoundError(f"Dataset folder not found: {cfg_data_root}")

all_classes = sorted([p.name for p in cfg_data_root.iterdir() if p.is_dir()])
if cfg_normal_class not in all_classes:
    raise ValueError(f"{cfg_normal_class} not found in {all_classes}")

anomaly_classes = [c for c in all_classes if c != cfg_normal_class]
print("Dataset root:", cfg_data_root)
print("Classes:", all_classes)
print("Normal class:", cfg_normal_class)
print("Anomaly classes:", anomaly_classes)

In [ ]:
def list_images(folder: Path) -> list[Path]:
    exts = {".jpg", ".jpeg", ".png"}
    return sorted([p for p in folder.rglob("*") if p.suffix.lower() in exts])


def split_three_way(paths: list[Path], train_frac: float = 0.7, val_frac: float = 0.15, seed: int = 42):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(paths))
    rng.shuffle(idx)
    paths = [paths[i] for i in idx]

    n = len(paths)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    train = paths[:n_train]
    val = paths[n_train:n_train + n_val]
    test = paths[n_train + n_val:]
    return train, val, test


normal_paths = list_images(cfg_data_root / cfg_normal_class)
normal_train_paths, normal_val_paths, normal_test_paths = split_three_way(normal_paths, 0.7, 0.15, SEED)

anomaly_test_paths: dict[str, list[Path]] = {}
for cls in anomaly_classes:
    cls_paths = list_images(cfg_data_root / cls)
    _, _, cls_test = split_three_way(cls_paths, 0.7, 0.15, SEED)
    anomaly_test_paths[cls] = cls_test

print("Sunflower train/val/test:", len(normal_train_paths), len(normal_val_paths), len(normal_test_paths))
for cls in anomaly_classes:
    print(f"{cls} test:", len(anomaly_test_paths[cls]))

In [ ]:
def decode_and_resize(path: tf.Tensor, img_size: tuple[int, int]):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, img_size)
    img = tf.cast(img, tf.float32) / 255.0
    img.set_shape((img_size[0], img_size[1], 3))
    return img


def make_autoencoder_ds(paths: list[Path], img_size: tuple[int, int], batch_size: int, shuffle: bool):
    ds = tf.data.Dataset.from_tensor_slices([str(p) for p in paths])
    if shuffle:
        ds = ds.shuffle(buffer_size=max(1000, len(paths)), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p: decode_and_resize(p, img_size), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x: (x, x), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


train_ds = make_autoencoder_ds(normal_train_paths, cfg_img_size, cfg_batch_size, shuffle=True)
val_ds = make_autoencoder_ds(normal_val_paths, cfg_img_size, cfg_batch_size, shuffle=False)

print(train_ds)
print(val_ds)

In [ ]:
def build_conv_autoencoder(img_size: tuple[int, int]):
    inp = tf.keras.Input(shape=(img_size[0], img_size[1], 3), name="image")

    # Encoder: Conv + MaxPool
    x = tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same")(inp)
    x = tf.keras.layers.MaxPooling2D(2, padding="same")(x)
    x = tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = tf.keras.layers.MaxPooling2D(2, padding="same")(x)
    x = tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    encoded = tf.keras.layers.MaxPooling2D(2, padding="same", name="encoded")(x)

    # Decoder: Transposed conv upsampling
    x = tf.keras.layers.Conv2DTranspose(128, 3, strides=2, activation="relu", padding="same")(encoded)
    x = tf.keras.layers.Conv2DTranspose(64, 3, strides=2, activation="relu", padding="same")(x)
    x = tf.keras.layers.Conv2DTranspose(32, 3, strides=2, activation="relu", padding="same")(x)
    out = tf.keras.layers.Conv2D(3, 3, activation="sigmoid", padding="same")(x)

    autoencoder = tf.keras.Model(inp, out, name="conv_autoencoder")
    encoder = tf.keras.Model(inp, encoded, name="encoder")
    return autoencoder, encoder


autoencoder, encoder = build_conv_autoencoder(cfg_img_size)
autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=cfg_learning_rate),
    loss="mse",
)
autoencoder.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
]

history = autoencoder.fit(
    train_ds,
    validation_data=val_ds,
    epochs=cfg_epochs,
    callbacks=callbacks,
    verbose=1,
)

plt.figure(figsize=(6, 4))
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.title("Sunflower Autoencoder Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.tight_layout()

## Requirement 1: Original vs Reconstructed (Sunflower)

In [ ]:
batch = next(iter(val_ds))[0]
recon = autoencoder.predict(batch[:6], verbose=0)

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    axes[0, i].imshow(batch[i])
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")

    axes[1, i].imshow(np.clip(recon[i], 0, 1))
    axes[1, i].set_title("Reconstructed")
    axes[1, i].axis("off")

plt.tight_layout()

## Requirement 2: MSE for Normal vs Anomalous Flower Types

In [ ]:
def load_array(paths: list[Path], img_size: tuple[int, int]) -> np.ndarray:
    arr = []
    for p in paths:
        b = tf.io.read_file(str(p))
        img = tf.image.decode_image(b, channels=3, expand_animations=False)
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32) / 255.0
        arr.append(img.numpy())
    return np.asarray(arr, dtype=np.float32)


def reconstruction_mse(model: tf.keras.Model, x: np.ndarray, batch_size: int = 64) -> np.ndarray:
    recon = model.predict(x, batch_size=batch_size, verbose=0)
    return np.mean((x - recon) ** 2, axis=(1, 2, 3))


x_norm_test = load_array(normal_test_paths, cfg_img_size)
mse_by_class: dict[str, np.ndarray] = {}

mse_by_class[cfg_normal_class] = reconstruction_mse(autoencoder, x_norm_test)
for cls in anomaly_classes:
    x_cls = load_array(anomaly_test_paths[cls], cfg_img_size)
    mse_by_class[cls] = reconstruction_mse(autoencoder, x_cls)

summary_rows = []
for cls, values in mse_by_class.items():
    summary_rows.append(
        {
            "class": cls,
            "mean_mse": float(np.mean(values)),
            "std_mse": float(np.std(values)),
            "median_mse": float(np.median(values)),
            "n": len(values),
        }
    )

mse_summary_df = pd.DataFrame(summary_rows).sort_values("mean_mse")
mse_summary_df

In [ ]:
plt.figure(figsize=(8, 4.5))
for cls in [cfg_normal_class] + anomaly_classes:
    plt.hist(mse_by_class[cls], bins=35, alpha=0.5, label=cls)
plt.title("Reconstruction MSE Distribution by Flower Type")
plt.xlabel("MSE")
plt.ylabel("count")
plt.legend()
plt.tight_layout()

## Requirement 3: Density Score Distributions (Train/Val/Anomalies)

In [ ]:
x_train = load_array(normal_train_paths, cfg_img_size)
x_val = load_array(normal_val_paths, cfg_img_size)

z_train = encoder.predict(x_train, batch_size=64, verbose=0).reshape(len(x_train), -1)
z_val = encoder.predict(x_val, batch_size=64, verbose=0).reshape(len(x_val), -1)

kde = KernelDensity(kernel="gaussian", bandwidth=0.5)
kde.fit(z_train)

density_scores: dict[str, np.ndarray] = {
    "sunflower_train": kde.score_samples(z_train),
    "sunflower_val": kde.score_samples(z_val),
}

for cls in anomaly_classes:
    x_cls = load_array(anomaly_test_paths[cls], cfg_img_size)
    z_cls = encoder.predict(x_cls, batch_size=64, verbose=0).reshape(len(x_cls), -1)
    density_scores[cls] = kde.score_samples(z_cls)

density_rows = []
for name, vals in density_scores.items():
    density_rows.append(
        {
            "split_or_class": name,
            "mean_log_density": float(np.mean(vals)),
            "std_log_density": float(np.std(vals)),
            "median_log_density": float(np.median(vals)),
            "n": len(vals),
        }
    )

density_summary_df = pd.DataFrame(density_rows).sort_values("mean_log_density", ascending=False)
density_summary_df

In [ ]:
# Plot distribution against train/val for each anomaly class.
for cls in anomaly_classes:
    plt.figure(figsize=(7.5, 4.2))
    plt.hist(density_scores["sunflower_train"], bins=40, alpha=0.45, label="sunflower_train")
    plt.hist(density_scores["sunflower_val"], bins=40, alpha=0.45, label="sunflower_val")
    plt.hist(density_scores[cls], bins=40, alpha=0.55, label=f"{cls}_anomaly")
    plt.title(f"Density Scores: sunflower train/val vs {cls}")
    plt.xlabel("log density (KDE on latent vectors)")
    plt.ylabel("count")
    plt.legend()
    plt.tight_layout()

## Requirement 4: How the Autoencoder Works for Anomaly Detection

The autoencoder is trained only on sunflower images, so it learns a compact latent representation specialized for sunflower structure, color, and texture. During inference, sunflower images are typically reconstructed with lower error, while non-sunflower flowers are reconstructed less accurately, producing higher MSE. This reconstruction gap is used as an anomaly signal.

Using latent-space density adds a second anomaly signal: if an encoded sample falls in a low-density region compared with sunflower training encodings, it is likely novel/anomalous. In this notebook, anomaly-like classes generally show lower log-density than sunflower train/validation encodings.

## Requirement 5: Other Methods Besides Autoencoders

- **One-Class SVM:** Learns a boundary around normal data and flags outside points as anomalies.
- **Isolation Forest:** Random partitioning isolates anomalies faster because they are rare and different.
- **Local Outlier Factor (LOF):** Uses local neighborhood density to detect points with unusually low local density.
- **Deep SVDD / One-Class Deep Methods:** Trains a network to map normal data near a compact center.
- **GAN-based anomaly detection (e.g., AnoGAN variants):** Uses generator/discriminator mismatch as anomaly evidence.
- **Diffusion or energy-based methods:** Estimates normal data likelihood/energy and flags unlikely samples.

## Acknowledgment

I used a coding assistant (GPT-5.3-Codex) to help scaffold and organize this notebook.